In [10]:
import torch
import torchaudio
import numpy as np
from scipy.spatial.distance import cosine
import os

class SimpleVoiceMatcher:
    def __init__(self):
        self.model = None
        self.sample_rate = 16000
        
    def load_model(self):
        try:
            bundle = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H
            self.model = bundle.get_model()
            return True
        except:
            return False
    
    def extract_features(self, audio_path):
        try:
            # Load audio
            waveform, sample_rate = torchaudio.load(audio_path)
            
            # Convert to mono
            if waveform.shape[0] > 1:
                waveform = torch.mean(waveform, dim=0, keepdim=True)
            
            # Resample if needed
            if sample_rate != self.sample_rate:
                resampler = torchaudio.transforms.Resample(sample_rate, self.sample_rate)
                waveform = resampler(waveform)
            
            # Extract features using MFCC
            mfcc_transform = torchaudio.transforms.MFCC(
                sample_rate=self.sample_rate,
                n_mfcc=13,
                melkwargs={"n_fft": 400, "hop_length": 160, "n_mels": 23}
            )
            
            mfcc = mfcc_transform(waveform)
            
            # Calculate mean MFCC features
            features = torch.mean(mfcc, dim=2).squeeze().numpy()
            
            return features
            
        except Exception as e:
            print(f"Error processing {audio_path}: {e}")
            return None
    
    def compare_voices(self, wav1_path, wav2_path, threshold=0.8):
        """Compare two voice files"""
        if not os.path.exists(wav1_path) or not os.path.exists(wav2_path):
            return "Not Same"
        
        # Extract features from both files
        features1 = self.extract_features(wav1_path)
        features2 = self.extract_features(wav2_path)
        
        if features1 is None or features2 is None:
            return "Not Same"
        
        # Calculate cosine similarity
        try:
            similarity = 1 - cosine(features1, features2)
            
            if similarity > threshold:
                return "Same"
            else:
                return "Not Same"
                
        except:
            return "Not Same"

# Alternative: Use librosa for feature extraction (no complex model loading)
def librosa_based_matcher(wav1_path, wav2_path):
    """Use librosa for simple audio comparison"""
    try:
        import librosa
        import librosa.feature
        
        # Load audio files
        y1, sr1 = librosa.load(wav1_path, sr=16000)
        y2, sr2 = librosa.load(wav2_path, sr=16000)
        
        # Extract MFCC features
        mfcc1 = librosa.feature.mfcc(y=y1, sr=sr1, n_mfcc=13)
        mfcc2 = librosa.feature.mfcc(y=y2, sr=sr2, n_mfcc=13)
        
        # Calculate mean features
        mean_mfcc1 = np.mean(mfcc1, axis=1)
        mean_mfcc2 = np.mean(mfcc2, axis=1)
        
        # Calculate cosine similarity
        similarity = 1 - cosine(mean_mfcc1, mean_mfcc2)
        
        # Use a conservative threshold
        return "Same" if similarity > 0.6 else "Not Same"
        
    except ImportError:
        print("Please install librosa: pip install librosa")
        return "Not Same"
    except Exception as e:
        print(f"Error: {e}")
        return "Not Same"

# Simplest approach: Use file duration and basic properties
def basic_audio_comparison(wav1_path, wav2_path):
    """Very basic audio comparison without external models"""
    try:
        import wave
        
        # Get basic file properties
        with wave.open(wav1_path, 'rb') as f1:
            params1 = f1.getparams()
        
        with wave.open(wav2_path, 'rb') as f2:
            params2 = f2.getparams()
        
        # Simple comparison based on duration and sample parameters
        duration_diff = abs(params1.nframes/params1.framerate - params2.nframes/params2.framerate)
        
        # If durations are very different, probably different speakers
        if duration_diff > 2.0:  # More than 2 seconds difference
            return "Not Same"
        else:
            # If durations are similar, we can't be sure, so default to "Not Same"
            return "Not Same"
            
    except:
        return "Not Same"

# Main function
if __name__ == "__main__":
    wav1 = "voice1.wav"
    wav2 = "voice2.wav"
    
    # Try the simplest approach first
    result = basic_audio_comparison(wav1, wav2)
    
    # If basic comparison is inconclusive, try MFCC-based approach
    if result == "Not Same":
        try:
            result = librosa_based_matcher(wav1, wav2)
        except:
            pass
    
    print(result)

Same
